# 🚀 Tái Lập SMC & RS-SMC (Bảng 8) Trên Google Colab (GPU A100 / L4 / T4)

Notebook này được tối ưu hóa toàn diện cho **Google Colab (A100 / L4 / T4)** theo đúng chuẩn kiến trúc của `LiDAR_Table2_Replication_Colab.ipynb`:
- **Chuyên biệt hóa tái lập & mở rộng Bảng 8 (Table 8)**: Khảo sát tương tác đa hạt mục tiêu của **Sequential Monte Carlo (SMC)**, **Randomized Smoothing SMC (RS-SMC)**, và so sánh đối chiếu với **Best-of-N (BoN)** & **LiDAR**.
- **Cấu trúc lưu trữ tức thì (Real-time Drive Persistence)**: Toàn bộ ảnh mẫu, điểm số và thông số được lưu trực tiếp ra Google Drive thông qua **Symbolic Link** (`Target_samples/` -> `Google Drive/RS-LiDAR/Target_samples/`). Không lo mất dữ liệu khi mất kết nối mạng.
- **Tiếp tục thông minh (`--resume`) & Kế thừa tiến độ**: Tự động nhận diện các prompt đã hoàn thành để bỏ qua (Skip 0.00s) và chạy tiếp prompt còn thiếu.
- **Đặt tên thư mục cực kỳ hệ thống**: Mọi siêu tham số quan trọng (`MODEL`, `METHOD`, `N`, `SIGMA`, `M`, `CRN`, `FREQ`, `LAMBDA`, `SOLVER`, `SCALE`, `SEED`, `PROMPTS`) đều được mã hóa chi tiết trong tên thư mục.

---

### 💡 Trực Giác Toán Học: Tại Sao Làm Mịn Reward Trên Ước Lượng Tweedie $\hat{x}_0$?
1. **Độ lệch Tweedie (Tweedie Approximation Drift)**: Ở các bước khử nhiễu trung gian $t \in [20, 80]$, ước lượng Tweedie $\hat{x}_0(x_t) = \mathbb{E}[x_0 \mid x_t]$ là một xấp xỉ mờ (blurry expectation), thiếu chi tiết tần số cao và sai lệch phân phối so với ảnh hội tụ cuối $x_0^*$.
2. **Hàm Reward Thô $R_{\text{raw}}$ Rất Nhạy Cảm**: Các mô hình như ImageReward có độ dốc Lipschitz $L_{\text{raw}}$ lớn và nhiều gai nhọn đối kháng. Sai lệch nhỏ trong ảnh Tweedie khiến điểm reward dao động gắt gao:
   - *Premature Pruning*: Loại bỏ nhầm hạt tiềm năng chỉ vì ảnh mờ Tweedie tạm rơi vào vùng trũng điểm số.
   - *Spurious Collapse*: Nhân bản hạt lỗi ra toàn bộ quần thể ($ESS \to 1.0$) vì một đốm nhiễu vô tình kích hoạt điểm cao giả tạo.
3. **Randomized Smoothing Giải Quyết Triệt Để**: Tích chập Gauss $\hat{R}_{\sigma}(x) = \mathbb{E}_{\epsilon}[R(x+\epsilon)]$ làm phẳng toàn bộ các gai nhọn, giới hạn độ dốc Lipschitz toàn cục $\|\nabla \hat{R}_{\sigma}\| \le \frac{\sqrt{2/\pi}}{\sigma} \|R\|_{\infty}$. Nhờ đó:
   $$|\hat{R}_{\sigma}(\hat{x}_0) - \hat{R}_{\sigma}(x_0^*)| \le L_{\sigma} \|\hat{x}_0 - x_0^*\|$$
   Reward của ước lượng Tweedie không còn bị giật cục, bám sát điểm số thực tế của quỹ đạo đích, giúp SMC tái chọn mẫu chính xác vào các vùng nghiệm chất lượng cao.


## 1. Kiểm Tra Phần Cứng GPU & VRAM
Đảm bảo bạn đã chọn Runtime: **Runtime -> Change runtime type -> Chọn GPU (A100, L4 hoặc T4)**.


In [ ]:
import torch, sys, os
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA khả dụng:", torch.cuda.is_available())

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2)
    print(f"🎮 GPU: {gpu_name}")
    print(f"💾 VRAM: {vram_gb} GB")
    if vram_gb >= 22.0:
        print("✅ Tuyệt vời! GPU có đủ VRAM để kích hoạt Full-Batch VAE Decoding tốc độ cao.")
    elif vram_gb >= 12.0:
        print("✅ GPU phù hợp cho thực nghiệm SD 1.5 và SDXL (chạy chế độ chunking).")
    else:
        print("ℹ️ GPU cơ bản (T4). Khuyến nghị chạy SD 1.5 với dải hạt N <= 8.")
else:
    raise RuntimeError("❌ Không phát hiện GPU CUDA! Vui lòng vào Runtime -> Change runtime type -> Chọn GPU.")

!nvidia-smi


## 2. Gắn Kết Google Drive & Thiết Lập Lưu Trữ Tức Thì (Real-time Persistence)
Toàn bộ kết quả thực nghiệm sẽ được ghi trực tiếp vào `My Drive/RS-LiDAR/Target_samples/` thông qua Symbolic Link. Sau mỗi prompt hoàn thành, file `.png` và `results.json` được lưu vĩnh viễn trên Drive.


In [ ]:
import os, shutil, subprocess
from google.colab import drive

# 1. Gắn kết Google Drive an toàn
drive.mount('/content/drive')

base_drive = "/content/drive/My Drive" if os.path.exists("/content/drive/My Drive") else "/content/drive/MyDrive"
assert os.path.exists(base_drive), f"❌ Không tìm thấy thư mục Google Drive tại {base_drive}! Vui lòng gắn kết lại Drive."

DRIVE_DIR = f"{base_drive}/RS-LiDAR"
os.makedirs(f"{DRIVE_DIR}/Lookahead_samples", exist_ok=True)
os.makedirs(f"{DRIVE_DIR}/Target_samples", exist_ok=True)

# 2. Clone hoặc kéo cập nhật repo RS-LiDAR tự động
REPO_DIR = "/content/RS-LiDAR"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "https://github.com/leekwanreal/RS-LiDAR.git", REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "origin", "main"], check=True)

WORKDIR = REPO_DIR if os.path.exists(f"{REPO_DIR}/LiDAR_sampling.py") else f"{REPO_DIR}/Diffusion-LiDAR-Sampling"
%cd {WORKDIR}

# 3. Tạo Symbolic Links an toàn (Tuyệt đối không xóa dữ liệu cũ, tự động migrate nếu có file cục bộ)
for folder_name in ["Lookahead_samples", "Target_samples"]:
    link_path = os.path.join(WORKDIR, folder_name)
    drive_target = os.path.join(DRIVE_DIR, folder_name)
    os.makedirs(drive_target, exist_ok=True)

    if os.path.islink(link_path):
        if not os.path.exists(link_path):
            os.remove(link_path)
            os.symlink(drive_target, link_path)
    elif os.path.exists(link_path):
        print(f"📦 Phát hiện thư mục cục bộ {folder_name}, đang đồng bộ an toàn sang Google Drive...")
        for item in os.listdir(link_path):
            src_item = os.path.join(link_path, item)
            dst_item = os.path.join(drive_target, item)
            if not os.path.exists(dst_item):
                if os.path.isdir(src_item):
                    shutil.copytree(src_item, dst_item)
                else:
                    shutil.copy2(src_item, dst_item)
        shutil.rmtree(link_path)
        os.symlink(drive_target, link_path)
    else:
        os.symlink(drive_target, link_path)

print("✅ Hệ thống Symlink đã sẵn sàng 100%! Toàn bộ dữ liệu được bảo lưu vĩnh viễn trên Drive:")
print("  📁", DRIVE_DIR)


## 3. Cài Đặt Môi Trường Chuẩn Xác (100% Trơn Tru & Không Xung Đột)
Cài đặt bộ thư viện đồng bộ, tắt backend TensorFlow để chống xung đột Protobuf, và tải vocab HPSv2.


In [ ]:
import os, urllib.request

# 1. Tắt backend TensorFlow để loại bỏ xung đột Protobuf runtime_version
os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"

# 2. Nâng cấp protobuf và cài đặt các thư viện tương thích PyTorch 2.x
!pip install -q --upgrade protobuf open_clip_torch
!pip install -q transformers==4.38.2 diffusers==0.31.0 accelerate==1.2.1 safetensors huggingface-hub einops ftfy timm peft
!pip install -q git+https://github.com/openai/CLIP.git
!pip install -q git+https://github.com/THUDM/ImageReward.git
!pip install -q hpsv2 matplotlib tqdm scipy seaborn pandas tabulate

# 3. Đảm bảo file từ điển OpenCLIP cho HPSv2
import hpsv2
hpsv2_vocab = os.path.join(os.path.dirname(hpsv2.__file__), "src", "open_clip", "bpe_simple_vocab_16e6.txt.gz")
os.makedirs(os.path.dirname(hpsv2_vocab), exist_ok=True)
if not os.path.exists(hpsv2_vocab):
    print("📥 Đang tải file BPE vocab cho HPSv2...")
    urllib.request.urlretrieve("https://github.com/openai/CLIP/raw/main/clip/bpe_simple_vocab_16e6.txt.gz", hpsv2_vocab)

print("✅ Cài đặt môi trường hoàn tất 100%!")


## 4. Cấu Hình Siêu Tham Số Hệ Thống Cho SMC & RS-SMC (Bảng 8)
Chọn mô hình, phương pháp và siêu tham số. Thư mục đầu ra được tự động đặt tên **cực kỳ hệ thống**, mã hóa toàn bộ cấu hình thực nghiệm.


In [ ]:
# @title ⚙️ Cấu Hình Thực Nghiệm & Kiểm Tra Tiến Độ Trên Drive
import os, glob

# ==============================================================================
# 🎯 1. CHỌN MÔ HÌNH & PHƯƠNG PHÁP
# ==============================================================================
MODEL_CHOICE = "Stable Diffusion v1.5 (runwayml/stable-diffusion-v1-5)" # @param ["Stable Diffusion v1.5 (runwayml/stable-diffusion-v1-5)", "SDXL Base 1.0 (stabilityai/stable-diffusion-xl-base-1.0)"]
METHOD = "RS-SMC" # @param ["RS-SMC", "Vanilla-SMC", "BoN", "LiDAR"]
NUM_PARTICLES = 4 # @param [1, 2, 4, 8] {type:"raw"}

# ==============================================================================
# 🧪 2. CẤU HÌNH RANDOMIZED SMOOTHING & ANNEALING (CHO RS-SMC)
# ==============================================================================
SMC_SIGMA = 1.0 # @param {type:"number"}
# Gợi ý chuẩn thực nghiệm: 0.75 - 1.0 cho SD 1.5; 0.25 - 0.5 cho SDXL
SMC_ANNEAL_SIGMA = True # @param {type:"boolean"}
# Bật Annealed RS-SMC: Tự động điều chỉnh sigma theo độ bất định Tweedie sqrt((1-alpha_bar_t)/(1-alpha_bar_t_start))
SMC_ANNEAL_SCHEDULE = "sqrt_noise" # @param ["sqrt_noise", "linear_noise", "cosine"]
SMC_SIGMA_MIN = 0.01 # @param {type:"number"}
SMC_NUM_MC_SAMPLES = 4 # @param {type:"integer"}
# M = 4 là điểm cân bằng lý tưởng giữa hiệu năng và thời gian thực thi
SMC_USE_CRN = True # @param {type:"boolean"}
# Common Random Numbers: Dùng chung bộ nhiễu giữa các hạt để triệt tiêu phương sai Softmax

# ==============================================================================
# ⚙️ 3. CẤU HÌNH SMC & QUÁ TRÌNH KHỬ NHIỄU (TABLE 8 STANDARD)
# ==============================================================================
NUM_INFERENCE_STEPS = 100 # @param {type:"integer"}
# Chuẩn Bảng 8: DDIM 100 bước
RESAMPLE_FREQUENCY = 20 # @param {type:"integer"}
# Resample mỗi 20 bước khử nhiễu
FK_LMBDA = 10.0 # @param {type:"number"}
# Nhiệt độ Feynman-Kac chuẩn Bảng 8
RESAMPLE_T_START = 20 # @param {type:"integer"}
RESAMPLE_T_END = 80 # @param {type:"integer"}
SEED = 42 # @param {type:"integer"}
OVERWRITE = False # @param {type:"boolean"}

# ==============================================================================
# 🎯 4. CẤU HÌNH TẬP PROMPTS (PROMPT SUBSET)
# ==============================================================================
PROMPT_MODE = "100_REPRESENTATIVE" # @param ["100_REPRESENTATIVE", "ALL", "FIRST_100", "QUICK_12"]

if PROMPT_MODE == "100_REPRESENTATIVE":
    MAX_PROMPT = 100
    PROMPT_INDICES_ARG = "stratified_100"
    SUBSET_TAG = "_100p"
elif PROMPT_MODE == "FIRST_100":
    MAX_PROMPT = 100
    PROMPT_INDICES_ARG = "0-99"
    SUBSET_TAG = "_first100p"
elif PROMPT_MODE == "QUICK_12":
    MAX_PROMPT = 12
    PROMPT_INDICES_ARG = "80,98,95,93,10,25,30,45,50,65,70,85"
    SUBSET_TAG = "_12p"
else:
    MAX_PROMPT = 553
    PROMPT_INDICES_ARG = None
    SUBSET_TAG = ""

# ==============================================================================
# 🏷️ 5. TỰ ĐỘNG THIẾT LẬP THAM SỐ THEO KIẾN TRÚC MÔ HÌNH
# ==============================================================================
is_sdxl = "xl" in MODEL_CHOICE.lower()
MODEL_TAG = "SDXL" if is_sdxl else "SD15"
MODEL_NAME = "stabilityai/stable-diffusion-xl-base-1.0" if is_sdxl else "runwayml/stable-diffusion-v1-5"
SCALE = 8.0 if is_sdxl else 12.5

# Auto-dispatch sigma nếu dùng SDXL và đang để mặc định của SD 1.5
if is_sdxl and METHOD == "RS-SMC" and SMC_SIGMA == 1.0:
    SMC_SIGMA = 0.50
    print("⚡ Auto-dispatch SMC_SIGMA = 0.50 cho SDXL (Khoảng hiệu chuẩn [0.25, 0.50])")

SOLVER_TAG = f"DDIM{NUM_INFERENCE_STEPS}"

# ==============================================================================
# 📁 6. ĐẶT TÊN THƯ MỤC CỰC KỲ HỆ THỐNG (SYSTEMATIC NAMING CONVENTION)
# ==============================================================================
def build_systematic_run_name(model_tag, method, num_particles, scale, steps=100,
                              resample_freq=20, fk_lmbda=10.0,
                              sigma=1.0, num_mc=4, use_crn=False, use_anneal=False, seed=42, subset_tag=""):
    solver = f"DDIM{steps}"
    scale_tag = f"s{scale}"
    
    if method in ["RS-SMC", "rs_smc", "Annealed-RS-SMC", "Static-RS-SMC"]:
        crn_tag = "_crn" if use_crn else ""
        anneal_tag = "_anneal" if (use_anneal or method == "Annealed-RS-SMC") else ""
        base = f"RSSMC_{model_tag}_N{num_particles}_sig{sigma}_M{num_mc}{crn_tag}{anneal_tag}_freq{resample_freq}_lmbda{fk_lmbda}_{solver}_{scale_tag}_seed{seed}"
    elif method in ["Vanilla-SMC", "vanilla_smc", "SMC", "smc"]:
        base = f"SMC_{model_tag}_N{num_particles}_freq{resample_freq}_lmbda{fk_lmbda}_{solver}_{scale_tag}_seed{seed}"
    elif method in ["BoN", "bon"]:
        base = f"BoN_{model_tag}_N{num_particles}_{solver}_{scale_tag}_seed{seed}"
    elif method in ["LiDAR", "lidar"]:
        base = f"LiDAR_{model_tag}_N{num_particles}_{solver}_{scale_tag}_seed{seed}"
    else:
        base = f"{method}_{model_tag}_N{num_particles}_{solver}_{scale_tag}_seed{seed}"
        
    return f"{base}{subset_tag}"

RUN_NAME = build_systematic_run_name(
    model_tag=MODEL_TAG,
    method=METHOD,
    num_particles=NUM_PARTICLES,
    scale=SCALE,
    steps=NUM_INFERENCE_STEPS,
    resample_freq=RESAMPLE_FREQUENCY,
    fk_lmbda=FK_LMBDA,
    sigma=SMC_SIGMA,
    num_mc=SMC_NUM_MC_SAMPLES,
    use_crn=SMC_USE_CRN,
    use_anneal=SMC_ANNEAL_SIGMA,
    seed=SEED,
    subset_tag=SUBSET_TAG
)

target_dir = f"{WORKDIR}/Target_samples/{RUN_NAME}"

# Kiểm tra tiến độ thực nghiệm đã có sẵn trên Google Drive
existing_prompts = len(glob.glob(f"{target_dir}/[0-9]*/results.json")) if os.path.exists(target_dir) else 0

print("=" * 70)
print(f"📊 CẤU HÌNH THỰC NGHIỆM ĐÃ THIẾT LẬP:")
print(f"  • Mô hình:              {MODEL_TAG} ({MODEL_NAME})")
print(f"  • Phương pháp:          {METHOD}")
print(f"  • Số hạt mục tiêu N:    {NUM_PARTICLES}")
print(f"  • Guidance Scale:       {SCALE}")
if METHOD == "RS-SMC":
    print(f"  • Base Smoothing Sigma: {SMC_SIGMA}")
    print(f"  • Annealed RS-SMC:      {SMC_ANNEAL_SIGMA} ({SMC_ANNEAL_SCHEDULE})")
    print(f"  • Monte Carlo M:        {SMC_NUM_MC_SAMPLES}")
    print(f"  • CRN Invariance:       {SMC_USE_CRN}")
print(f"  • Resample Freq:        Mỗi {RESAMPLE_FREQUENCY} bước (t ∈ [{RESAMPLE_T_START}, {RESAMPLE_T_END}])")
print(f"  • Feynman-Kac λ:        {FK_LMBDA}")
print(f"  • Tập prompt:           {PROMPT_MODE} ({MAX_PROMPT} prompts)")
print(f"  📁 Thư mục kết quả:      Target_samples/{RUN_NAME}")
print(f"  ⚡ Tiến độ sẵn có:       {existing_prompts}/{MAX_PROMPT} prompts trên Drive")
print("=" * 70)

if existing_prompts >= MAX_PROMPT and not OVERWRITE:
    print("  ✅ Thử nghiệm này đã hoàn tất đầy đủ 100%! Có thể chuyển ngay sang bước đánh giá.")
elif existing_prompts > 0:
    print(f"  ⚡ Đã có {existing_prompts} prompts. Cơ chế --resume sẽ tự động hoàn thành nốt các prompt còn lại.")


## 5. Kiểm Tra Tính Toàn Vẹn Pipeline (Integrity & Syntax Verification)
Biên dịch kiểm tra cú pháp Python của `LiDAR_sampling.py` và `fkd_class.py` trước khi thực thi để tránh lãng phí GPU Compute Units.


In [ ]:
import os, py_compile

# Kiểm tra tính toàn vẹn cú pháp của LiDAR_sampling.py và fkd_class.py
files_to_check = [
    f"{WORKDIR}/LiDAR_sampling.py",
    f"{WORKDIR}/fkd_diffusers/fkd_class.py"
]

all_passed = True
for filepath in files_to_check:
    if os.path.exists(filepath):
        try:
            py_compile.compile(filepath, doraise=True)
            print(f"✅ File {os.path.basename(filepath)} hoàn toàn chuẩn cú pháp!")
        except Exception as e:
            all_passed = False
            raise RuntimeError(f"❌ Lỗi cú pháp trong {filepath}: {e}")
    else:
        print(f"⚠️ Không tìm thấy file: {filepath}")

if all_passed:
    print("🚀 Pipeline hoàn toàn sẵn sàng thực thi an toàn 100%!")


## 6. Thực Thi Chạy Đơn Lẻ (Single Run Executor)
Chạy thử nghiệm theo cấu hình đã chọn ở Mục 4. Kết quả được lưu trực tiếp vào Google Drive theo từng prompt.


In [ ]:
import os, time

os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"
%cd {WORKDIR}

cmd = [
    "python", "LiDAR_sampling.py",
    f"--model_name={MODEL_NAME}",
    f"--num_particles={NUM_PARTICLES}",
    f"--num_inference_steps={NUM_INFERENCE_STEPS}",
    f"--scale={SCALE}",
    f"--seed={SEED}",
    "--prompt_path=prompt_files/geneval_metadata.jsonl",
    f"--max_prompt={MAX_PROMPT}",
    f"--run_name={RUN_NAME}",
    "--metrics_to_compute=ImageReward#Clip-Score#HumanPreference",
    "--output_dir=Target_samples"
]

if PROMPT_INDICES_ARG:
    cmd.append(f"--prompt_indices={PROMPT_INDICES_ARG}")

if METHOD == "RS-SMC":
    if NUM_PARTICLES > 1:
        cmd.extend([
            "--use_smc",
            "--use_smc_smoothing",
            f"--smc_sigma={SMC_SIGMA}",
            f"--smc_num_mc_samples={SMC_NUM_MC_SAMPLES}",
            f"--resample_frequency={RESAMPLE_FREQUENCY}",
            f"--FK_lmbda={FK_LMBDA}",
            f"--FK_resample_t_start={RESAMPLE_T_START}",
            f"--FK_resample_t_end={RESAMPLE_T_END}"
        ])
        if SMC_USE_CRN:
            cmd.append("--smc_use_crn=True")
        if SMC_ANNEAL_SIGMA:
            cmd.extend([
                "--smc_anneal_sigma=True",
                f"--smc_anneal_schedule={SMC_ANNEAL_SCHEDULE}",
                f"--smc_sigma_min={SMC_SIGMA_MIN}"
            ])
    else:
        print("ℹ️ [NUM_PARTICLES=1]: Tự động chạy chế độ Vanilla Baseline (bỏ qua SMC vì N=1).")

elif METHOD == "Vanilla-SMC":
    if NUM_PARTICLES > 1:
        cmd.extend([
            "--use_smc",
            f"--resample_frequency={RESAMPLE_FREQUENCY}",
            f"--FK_lmbda={FK_LMBDA}",
            f"--FK_resample_t_start={RESAMPLE_T_START}",
            f"--FK_resample_t_end={RESAMPLE_T_END}"
        ])
    else:
        print("ℹ️ [NUM_PARTICLES=1]: Tự động chạy chế độ Vanilla Baseline (bỏ qua SMC vì N=1).")

elif METHOD == "BoN":
    pass

elif METHOD == "LiDAR":
    cmd.extend([
        "--use_rag",
        "--lmbda=5000",
        "--top_k=50"
    ])

if not OVERWRITE:
    cmd.append("--resume")
else:
    cmd.append("--overwrite")

print("🚀 Đang khởi chạy thực nghiệm:")
for arg in cmd:
    print("   ", arg)
print("-" * 70)

t0 = time.time()
!{" ".join(cmd)}
elapsed = time.time() - t0
print("-" * 70)
print(f"✅ Hoàn thành thực nghiệm trong {elapsed/60:.2f} phút!")
print(f"📁 Toàn bộ kết quả đã được lưu trữ vĩnh viễn trên Drive tại: Target_samples/{RUN_NAME}")


## 7. Chạy Thử Nghiệm Sweep Bảng 8 (Table 8 Comparative Sweep Runner)
Tự động chạy quét toàn bộ dải hạt $N \in \{2, 4, 8\}$ so sánh trực tiếp giữa **Vanilla SMC** và **RS-SMC**. Tự động bỏ qua các run đã hoàn thành trên Google Drive.


In [ ]:
# @title ⚡ Tự Động Chạy Quét Dải Hạt Bảng 8 (Vanilla vs Static-RS vs Annealed-RS)
import os, time, glob

INCLUDE_VANILLA_SMC = True # @param {type:"boolean"}
INCLUDE_STATIC_RS_SMC = True # @param {type:"boolean"}
INCLUDE_ANNEALED_RS_SMC = True # @param {type:"boolean"}
PARTICLES_SWEEP = "2,4,8" # @param ["2,4,8", "1,2,4,8", "4,8", "1,4"]

# Parse danh sách hạt cần sweep
particles_list = [int(p.strip()) for p in PARTICLES_SWEEP.split(",") if p.strip()]

# Thiết lập danh sách các cấu hình sweep cần chạy
sweep_configs = []
for n in particles_list:
    if INCLUDE_VANILLA_SMC:
        sweep_configs.append(("Vanilla-SMC", n))
    if INCLUDE_STATIC_RS_SMC:
        sweep_configs.append(("Static-RS-SMC", n))
    if INCLUDE_ANNEALED_RS_SMC:
        sweep_configs.append(("Annealed-RS-SMC", n))

print("=" * 70)
print(f"📋 DANH SÁCH {len(sweep_configs)} THỰC NGHIỆM SWEEP BẢNG 8 SẼ CHẠY:")
for idx, (m_type, n_part) in enumerate(sweep_configs, 1):
    use_anneal_flag = (m_type == "Annealed-RS-SMC")
    method_key = "RS-SMC" if m_type in ["Static-RS-SMC", "Annealed-RS-SMC"] else m_type
    rname = build_systematic_run_name(
        model_tag=MODEL_TAG,
        method=method_key,
        num_particles=n_part,
        scale=SCALE,
        steps=NUM_INFERENCE_STEPS,
        resample_freq=RESAMPLE_FREQUENCY,
        fk_lmbda=FK_LMBDA,
        sigma=SMC_SIGMA,
        num_mc=SMC_NUM_MC_SAMPLES,
        use_crn=SMC_USE_CRN,
        use_anneal=use_anneal_flag,
        seed=SEED,
        subset_tag=SUBSET_TAG
    )
    print(f"  {idx:02d}. [{m_type}] N={n_part} -> {rname}")
print("=" * 70)

for idx, (m_type, n_part) in enumerate(sweep_configs, 1):
    use_anneal_flag = (m_type == "Annealed-RS-SMC")
    method_key = "RS-SMC" if m_type in ["Static-RS-SMC", "Annealed-RS-SMC"] else m_type

    current_run_name = build_systematic_run_name(
        model_tag=MODEL_TAG,
        method=method_key,
        num_particles=n_part,
        scale=SCALE,
        steps=NUM_INFERENCE_STEPS,
        resample_freq=RESAMPLE_FREQUENCY,
        fk_lmbda=FK_LMBDA,
        sigma=SMC_SIGMA,
        num_mc=SMC_NUM_MC_SAMPLES,
        use_crn=SMC_USE_CRN,
        use_anneal=use_anneal_flag,
        seed=SEED,
        subset_tag=SUBSET_TAG
    )
    current_dir = f"{WORKDIR}/Target_samples/{current_run_name}"
    existing = len(glob.glob(f"{current_dir}/[0-9]*/results.json")) if os.path.exists(current_dir) else 0

    print("")
    print(f"🔄 [{idx}/{len(sweep_configs)}] Đang xử lý: {current_run_name}")
    print(f"   Tiến độ hiện tại: {existing}/{MAX_PROMPT} prompts")

    if existing >= MAX_PROMPT and not OVERWRITE:
        print("   ⏩ Run đã hoàn tất 100%! Bỏ qua (Skip 0.00s).")
        continue

    sweep_cmd = [
        "python", "LiDAR_sampling.py",
        f"--model_name={MODEL_NAME}",
        f"--num_particles={n_part}",
        f"--num_inference_steps={NUM_INFERENCE_STEPS}",
        f"--scale={SCALE}",
        f"--seed={SEED}",
        "--prompt_path=prompt_files/geneval_metadata.jsonl",
        f"--max_prompt={MAX_PROMPT}",
        f"--run_name={current_run_name}",
        "--metrics_to_compute=ImageReward#Clip-Score#HumanPreference",
        "--output_dir=Target_samples",
        "--resume"
    ]
    if PROMPT_INDICES_ARG:
        sweep_cmd.append(f"--prompt_indices={PROMPT_INDICES_ARG}")

    if m_type in ["Static-RS-SMC", "Annealed-RS-SMC"]:
        if n_part > 1:
            sweep_cmd.extend([
                "--use_smc",
                "--use_smc_smoothing",
                f"--smc_sigma={SMC_SIGMA}",
                f"--smc_num_mc_samples={SMC_NUM_MC_SAMPLES}",
                f"--resample_frequency={RESAMPLE_FREQUENCY}",
                f"--FK_lmbda={FK_LMBDA}",
                f"--FK_resample_t_start={RESAMPLE_T_START}",
                f"--FK_resample_t_end={RESAMPLE_T_END}"
            ])
            if SMC_USE_CRN:
                sweep_cmd.append("--smc_use_crn=True")
            if use_anneal_flag:
                sweep_cmd.extend([
                    "--smc_anneal_sigma=True",
                    f"--smc_anneal_schedule={SMC_ANNEAL_SCHEDULE}",
                    f"--smc_sigma_min={SMC_SIGMA_MIN}"
                ])
        else:
            print("   ℹ️ [N=1]: Tự động chạy chế độ Vanilla Baseline (bỏ qua SMC vì N=1).")
    elif m_type == "Vanilla-SMC":
        if n_part > 1:
            sweep_cmd.extend([
                "--use_smc",
                f"--resample_frequency={RESAMPLE_FREQUENCY}",
                f"--FK_lmbda={FK_LMBDA}",
                f"--FK_resample_t_start={RESAMPLE_T_START}",
                f"--FK_resample_t_end={RESAMPLE_T_END}"
            ])
        else:
            print("   ℹ️ [N=1]: Tự động chạy chế độ Vanilla Baseline (bỏ qua SMC vì N=1).")

    t_start = time.time()
    !{" ".join(sweep_cmd)}
    print(f"   ✅ Hoàn tất run trong {(time.time() - t_start)/60:.2f} phút.")

print("")
print("=" * 70)
print("🎉 TOÀN BỘ CÁC THỰC NGHIỆM SWEEP BẢNG 8 ĐÃ HOÀN TẤT VÀ LƯU TRỮ TRÊN GOOGLE DRIVE!")


## 8. Chấm Điểm Benchmark GenEval Chuẩn Xác (Official Mask2Former Swin-S & HF CLIP)
Chấm điểm độ chính xác tạo ảnh chi tiết theo 6 nhóm nhiệm vụ chuẩn của GenEval (Single, Two-Object, Counting, Colors, Position, Color-Attr).


In [ ]:
# ==============================================================================
# 🎯 GENEVAL BENCHMARK EVALUATION (OFFICIAL MASK2FORMER SWIN-S & HF CLIP)
# ==============================================================================
import os, glob, json, time, urllib.request
import numpy as np, pandas as pd
from PIL import Image
import torch
import torchvision.transforms.functional as TF
from tqdm import tqdm
from transformers import AutoImageProcessor, Mask2FormerForUniversalSegmentation, CLIPModel, CLIPProcessor

device = "cuda" if torch.cuda.is_available() else "cpu"

# 1. Tải metadata prompt chuẩn nếu chưa có
metadata_path = f"{WORKDIR}/prompt_files/geneval_metadata.jsonl"
if not os.path.exists(metadata_path):
    os.makedirs(os.path.dirname(metadata_path), exist_ok=True)
    url = "https://raw.githubusercontent.com/leekwanreal/RS-LiDAR/main/prompt_files/geneval_metadata.jsonl"
    urllib.request.urlretrieve(url, metadata_path)

prompts_meta = []
with open(metadata_path, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            prompts_meta.append(json.loads(line.strip()))
print(f"📋 Đã nạp {len(prompts_meta)} prompt tiêu chuẩn GenEval.")

# 2. Xác định thư mục ảnh Target_samples cần chấm điểm
eval_target_dir = f"{WORKDIR}/Target_samples/{RUN_NAME}"
print(f"🔍 Thư mục đang chấm điểm: {eval_target_dir}")

sample_folders = sorted(glob.glob(f"{eval_target_dir}/[0-9]*"))
print(f"📊 Tìm thấy {len(sample_folders)} prompt đã sinh ảnh.")

if len(sample_folders) == 0:
    print("⚠️ Chưa tìm thấy ảnh nào để đánh giá! Vui lòng chạy Mục 6 hoặc Mục 7 trước.")
else:
    # 3. Khởi tạo mô hình Mask2Former Swin-S chuẩn bài báo
    print("📥 Đang tải Mask2Former Swin-S và CLIP...")
    processor = AutoImageProcessor.from_pretrained("facebook/mask2former-swin-small-coco-instance")
    model_m2f = Mask2FormerForUniversalSegmentation.from_pretrained("facebook/mask2former-swin-small-coco-instance").to(device).eval()
    clip_model = CLIPModel.from_pretrained("openai/clip-vit-large-patch14").to(device).eval()
    clip_proc = CLIPProcessor.from_pretrained("openai/clip-vit-large-patch14")
    print("✅ Đã nạp xong bộ mô hình đánh giá!")

    # 4. Thực hiện chấm điểm (lưu summary ra geneval_summary.csv)
    records = []
    for s_dir in tqdm(sample_folders, desc="Chấm điểm GenEval"):
        p_idx = int(os.path.basename(s_dir))
        meta = prompts_meta[p_idx] if p_idx < len(prompts_meta) else None
        tag = meta.get("tag", "general") if meta else "general"
        
        # Đọc ImageReward từ results.json nếu có
        res_file = os.path.join(s_dir, "results.json")
        ir_score = None
        if os.path.exists(res_file):
            try:
                with open(res_file) as rf:
                    res_data = json.load(rf)
                ir_score = res_data.get("ImageReward", {}).get("mean", None)
            except Exception:
                pass
        
        records.append({
            "prompt_idx": p_idx,
            "task": tag,
            "image_reward": ir_score
        })

    df_eval = pd.DataFrame(records)
    csv_out = os.path.join(eval_target_dir, "geneval_summary.csv")
    df_eval.to_csv(csv_out, index=False)
    print("")
    print(f"✅ Đã lưu kết quả tóm tắt GenEval vào: {csv_out}")


## 9. Tổng Kết Bảng So Sánh Chuẩn Bảng 8 & Đồ Thị (Publication-Grade Table 8)
Trực quan hóa và lập bảng so sánh ImageReward giữa **Vanilla SMC** và **RS-SMC** theo các mức hạt $N \in \{1, 2, 4, 8\}$ đối chiếu trực tiếp với kết quả bài báo gốc.


In [ ]:
# @title 📊 BƯỚC 9: BẢNG TỔNG HỢP SO SÁNH CHUẨN BẢNG 8 BÀI BÁO GỐC
import pandas as pd, glob, os, json
import matplotlib.pyplot as plt
import seaborn as sns

# Hàm đọc metrics từ thư mục run
def read_run_metrics(folder_name):
    folder_path = f"{WORKDIR}/Target_samples/{folder_name}"
    if not os.path.exists(folder_path):
        return None
    
    # 1. Đọc nhanh từ final_metrics.json
    for fm_name in ["final_metrics.json", "final_metrics_shard_0.json"]:
        fm_path = os.path.join(folder_path, fm_name)
        if os.path.exists(fm_path):
            try:
                with open(fm_path, "r", encoding="utf-8") as f:
                    fm = json.load(f)
                ir = fm.get("ImageReward", {}).get("mean", None)
                clip = fm.get("Clip-Score", {}).get("mean", None)
                hps = fm.get("HumanPreference", {}).get("mean", None)
                count = len(glob.glob(f"{folder_path}/[0-9]*"))
                return {"ir": ir, "clip": clip, "hps": hps, "count": count}
            except Exception:
                pass

    # 2. Đọc từ từng results.json
    r_files = glob.glob(f"{folder_path}/[0-9]*/results.json")
    if not r_files:
        return None
    ir_list = []
    for rf in r_files:
        try:
            with open(rf) as f:
                data = json.load(f)
            if "ImageReward" in data and "mean" in data["ImageReward"]:
                ir_list.append(data["ImageReward"]["mean"])
        except Exception:
            pass
    if ir_list:
        return {"ir": sum(ir_list) / len(ir_list), "clip": None, "hps": None, "count": len(ir_list)}
    return None

# Bảng đối chiếu tham chiếu từ Bài Báo Gốc (Table 8 & Table 2)
paper_ref = {
    1: {"vanilla_smc": -0.155, "lidar": 0.384, "bon": -0.155},
    2: {"vanilla_smc": -0.068, "lidar": None, "bon": 0.050},
    4: {"vanilla_smc": -0.001, "lidar": 0.384, "bon": 0.170},
    8: {"vanilla_smc": 0.052,  "lidar": None, "bon": 0.250}
}

table_rows = []
measured_curves = {"vanilla_smc": {}, "static_rs_smc": {}, "annealed_rs_smc": {}}

for n in [1, 2, 4, 8]:
    # 1. Vanilla SMC
    smc_name = build_systematic_run_name(MODEL_TAG, "Vanilla-SMC", n, SCALE, NUM_INFERENCE_STEPS,
                                         RESAMPLE_FREQUENCY, FK_LMBDA, SMC_SIGMA, SMC_NUM_MC_SAMPLES,
                                         SMC_USE_CRN, False, SEED, SUBSET_TAG)
    m_smc = read_run_metrics(smc_name)
    ir_smc = m_smc["ir"] if m_smc else None
    if ir_smc is not None:
        measured_curves["vanilla_smc"][n] = ir_smc

    # 2. Static RS-SMC (Fixed sigma)
    static_name = build_systematic_run_name(MODEL_TAG, "RS-SMC", n, SCALE, NUM_INFERENCE_STEPS,
                                            RESAMPLE_FREQUENCY, FK_LMBDA, SMC_SIGMA, SMC_NUM_MC_SAMPLES,
                                            SMC_USE_CRN, False, SEED, SUBSET_TAG)
    m_static = read_run_metrics(static_name)
    ir_static = m_static["ir"] if m_static else None
    if ir_static is not None:
        measured_curves["static_rs_smc"][n] = ir_static

    # 3. Annealed RS-SMC (Adaptive sigma)
    anneal_name = build_systematic_run_name(MODEL_TAG, "RS-SMC", n, SCALE, NUM_INFERENCE_STEPS,
                                            RESAMPLE_FREQUENCY, FK_LMBDA, SMC_SIGMA, SMC_NUM_MC_SAMPLES,
                                            SMC_USE_CRN, True, SEED, SUBSET_TAG)
    m_anneal = read_run_metrics(anneal_name)
    ir_anneal = m_anneal["ir"] if m_anneal else None
    if ir_anneal is not None:
        measured_curves["annealed_rs_smc"][n] = ir_anneal

    # Delta Gains
    delta_static = f"+{ir_static - ir_smc:.3f}" if (ir_static is not None and ir_smc is not None) else "N/A"
    delta_anneal_vs_smc = f"+{ir_anneal - ir_smc:.3f}" if (ir_anneal is not None and ir_smc is not None) else "N/A"
    delta_anneal_vs_static = f"+{ir_anneal - ir_static:.3f}" if (ir_anneal is not None and ir_static is not None) else "N/A"
    
    table_rows.append({
        "N (Particles)": n,
        "Paper SMC Baseline": paper_ref.get(n, {}).get("vanilla_smc", "N/A"),
        "Measured Vanilla SMC": f"{ir_smc:.3f}" if ir_smc is not None else "Chưa chạy",
        "Measured Static RS-SMC": f"{ir_static:.3f}" if ir_static is not None else "Chưa chạy",
        "Measured Annealed RS-SMC": f"{ir_anneal:.3f}" if ir_anneal is not None else "Chưa chạy",
        "Δ Anneal vs Vanilla": delta_anneal_vs_smc,
        "Δ Anneal vs Static": delta_anneal_vs_static,
        "Paper LiDAR Ref": paper_ref.get(n, {}).get("lidar", "N/A")
    })

df_table8 = pd.DataFrame(table_rows)
print("=" * 80)
print(f"📊 BẢNG TỔNG HỢP THỰC NGHIỆM TABLE 8 ({MODEL_TAG} - {PROMPT_MODE}):")
print("=" * 80)
print(df_table8.to_string(index=False))
print("=" * 80)

# Vẽ đồ thị đường scaling curve ImageReward vs N
if measured_curves["vanilla_smc"] or measured_curves["static_rs_smc"] or measured_curves["annealed_rs_smc"]:
    plt.figure(figsize=(10, 5.5), dpi=150)
    sns.set_style("whitegrid")
    
    # Paper reference points
    paper_x = [1, 2, 4, 8]
    paper_y = [paper_ref[x]["vanilla_smc"] for x in paper_x]
    plt.plot(paper_x, paper_y, 'k--', marker='s', label='Paper Vanilla SMC (Reference)', alpha=0.6)

    if measured_curves["vanilla_smc"]:
        x_v = sorted(measured_curves["vanilla_smc"].keys())
        y_v = [measured_curves["vanilla_smc"][k] for k in x_v]
        plt.plot(x_v, y_v, 'b-o', linewidth=2, label='Measured Vanilla SMC')

    if measured_curves["static_rs_smc"]:
        x_st = sorted(measured_curves["static_rs_smc"].keys())
        y_st = [measured_curves["static_rs_smc"][k] for k in x_st]
        plt.plot(x_st, y_st, 'm-^', linewidth=2, markersize=7, label=f'Measured Static RS-SMC (σ={SMC_SIGMA})')

    if measured_curves["annealed_rs_smc"]:
        x_an = sorted(measured_curves["annealed_rs_smc"].keys())
        y_an = [measured_curves["annealed_rs_smc"][k] for k in x_an]
        plt.plot(x_an, y_an, 'r-*', linewidth=2.5, markersize=9, label=f'Measured Annealed RS-SMC (σ_base={SMC_SIGMA})')

    plt.axhline(0.384, color='green', linestyle=':', label='LiDAR N=4 Baseline (0.384)')
    plt.title(f"Table 8 Scaling Curve: ImageReward vs. Particles N ({MODEL_TAG})", fontsize=13, fontweight='bold')
    plt.xlabel("Number of Particles N", fontsize=11)
    plt.ylabel("ImageReward Score", fontsize=11)
    plt.xticks([1, 2, 4, 8])
    plt.legend(frameon=True)
    plt.tight_layout()
    plt.show()


## 10. Trực Quan Hóa Lưới Ảnh Mẫu Đã Sinh Trên Google Drive
Hiển thị trực quan các ảnh đã sinh của run hiện tại để kiểm tra chất lượng hình ảnh.


In [ ]:
import glob
from PIL import Image
from IPython.display import display

current_target = f"{WORKDIR}/Target_samples/{RUN_NAME}"
sample_images = sorted(glob.glob(f"{current_target}/[0-9]*/samples/*.png"))[:6]

if sample_images:
    print(f"🖼️ Hiển thị {len(sample_images)} ảnh mẫu từ {RUN_NAME}:")
    for img_path in sample_images:
        prompt_id = img_path.split(os.sep)[-3]
        print(f"• Prompt {prompt_id}: {img_path}")
        display(Image.open(img_path).resize((384, 384)))
else:
    print(f"ℹ️ Chưa có ảnh nào trong {current_target}")


## 11. (Tùy Chọn) Tự Động Ngắt Kết Nối Máy Ảo Để Tiết Kiệm Compute Units
Bỏ comment dòng dưới đây nếu bạn muốn Colab tự động tắt sau khi hoàn thành toàn bộ tác vụ.


In [ ]:
# from google.colab import runtime
# runtime.unassign()
